# Race Time Predictor

Prédiction du temps de course a partir d'une trace GPX.

**Architecture** : ce notebook est un orchestrateur pur — toute la logique est dans `core/race/`.

## 1. Imports

In [15]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from core.gpx_race import parse_gpx, segment_trace
from core.race import (
    MinettiModel, FatigueModel, RacePredictor,
    WeatherFetcher, WeatherModel,
    SplitsBuilder, PacerExporter,
    race_summary, format_time, format_pace, parse_hhmm,
)
from core.constants import SLOPE_COLORS

## 2. Parametres de la course

In [27]:
# --- Trace GPX ---
GPX_PATH  = '/home/gsainton/Downloads/ultra-trail-du-haut-giffre-2026-trail-du-mont-blanc-des-dames.gpx'
RACE_NAME = 'UTHG 2026'
RACE_DATE = '2026-06-21'           # YYYY-MM-DD
START_TIME = '04:00'               # HH:MM

# --- Modele de vitesse ---
V_FLAT_KMH     = 9.0
ATHLETE_FACTOR = 0.85

# --- Segmentation RDP ---
EPSILON_M  = 1.0
MIN_SEG_KM = 0.05

# --- Fatigue (calibre sur historique) ---
FATIGUE_A   = 0.753
FATIGUE_K   = 2.981
USE_FATIGUE = True

# --- Descente ---
V_DOWNHILL_MAX_KMH = 9.0

# --- Correction thermique ---
T_THRESHOLD = 15.0
ALPHA_HEAT  = 0.004
USE_THERMAL = True

# --- Ravitaillements : (nom, distance km) ---
RAVITOS = [
    ('Joux Plane',    18.3),
    ('Golèse',        30.0),
    ('Refuge Bostan', 38.1),
    ('Vallon',        46.6),
    ('Sixt',          59.2),
    ('Arrivée',       71.3),
]

# --- Barrières horaires : (nom, distance km, heure HH:MM ou None) ---
BARRIERES = [
    ('Joux Plane',    18.3, '8:00'),
    ('Golèse',        30.0, None),
    ('Refuge Bostan', 38.1, None),
    ('Vallon',        46.6, '16:00'),
    ('Sixt',          59.2, '20:00'),
    ('Arrivée',       71.3, '23:30'),
]


## 3. Chargement et segmentation GPX

In [28]:
df_gpx = parse_gpx(GPX_PATH)
df_seg = segment_trace(df_gpx, epsilon_m=EPSILON_M, min_seg_km=MIN_SEG_KM)

lat_c     = df_gpx['lat'].mean()
lon_c     = df_gpx['lon'].mean()
alt_ref   = float(df_gpx['ele'].mean())
start_min = parse_hhmm(START_TIME)

print(f"Trace : {df_gpx['dist_cum'].max() / 1000:.1f} km "
      f"| D+ {df_gpx['dplus'].max():.0f} m "
      f"| {len(df_seg)} segments")

Trace : 75.9 km | D+ 5052 m | 32 segments


## 4. Meteo et correction thermique

In [29]:
fetcher    = WeatherFetcher()
df_weather = fetcher.fetch(lat_c, lon_c, RACE_DATE) if USE_THERMAL else None

if df_weather is not None:
    print(df_weather[['time', 'temperature_2m', 'relative_humidity_2m',
                       'wbgt']].to_string(index=False))
    weather_model = WeatherModel(
        df_weather, start_min=start_min, alt_ref_m=alt_ref,
        t_threshold=T_THRESHOLD, alpha=ALPHA_HEAT,
    )
else:
    print('Previsions indisponibles - thermique desactive.')
    weather_model = None

               time  temperature_2m  relative_humidity_2m      wbgt
2026-06-21 00:00:00            18.4                    60 19.351324
2026-06-21 01:00:00            17.7                    62 18.899256
2026-06-21 02:00:00            17.2                    62 18.462952
2026-06-21 03:00:00            17.6                    58 18.496014
2026-06-21 04:00:00            18.3                    54 18.768804
2026-06-21 05:00:00            18.6                    51 18.771209
2026-06-21 06:00:00            18.7                    50 18.770214
2026-06-21 07:00:00            20.5                    50 20.289981
2026-06-21 08:00:00            22.2                    47 21.457205
2026-06-21 09:00:00            24.2                    38 22.158152
2026-06-21 10:00:00            25.7                    35 23.040343
2026-06-21 11:00:00            26.8                    33 23.691178
2026-06-21 12:00:00            27.7                    31 24.156646
2026-06-21 13:00:00            28.3             

## 5. Prediction par segment

Vitesse = V_flat x Minetti(pente) x athlete_factor x fatigue x thermique

In [30]:
predictor = RacePredictor(
    v_flat_kmh=V_FLAT_KMH,
    athlete_factor=ATHLETE_FACTOR,
    fatigue_model=FatigueModel(a=FATIGUE_A, k=FATIGUE_K),
    v_downhill_max_kmh=V_DOWNHILL_MAX_KMH,
)

df_pred = predictor.predict(
    df_seg, use_fatigue=USE_FATIGUE, weather_model=weather_model,
)

summary   = race_summary(df_pred, race_name=RACE_NAME)
end_min   = start_min + summary['total_min']

print(f"Course  : {summary['race_name']}")
print(f"Temps   : {summary['total_time_str']} "
      f"(allure moy. {summary['avg_pace_str']}/km)")
print(f"Arrivee : {int(end_min) // 60 % 24:02d}h{int(end_min) % 60:02d}")

Course  : UTHG 2026
Temps   : 15h 48min (allure moy. 12'30"/km)
Arrivee : 19h48


## 6. Profil altimetrique

In [31]:
dist_km = df_gpx['dist_cum'].values / 1000.0
ele     = df_gpx['ele'].values

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=dist_km, y=ele, fill='tozeroy',
    fillcolor='rgba(150,150,150,0.15)',
    line=dict(color='rgba(0,0,0,0)', width=0),
    showlegend=False, hoverinfo='skip',
))

legend_added = set()
for _, seg in df_pred.iterrows():
    cat   = seg['slope_category']
    color = SLOPE_COLORS.get(cat, '#888')
    mask  = (dist_km >= seg['dist_start_km']) & (dist_km <= seg['dist_end_km'])
    show_leg = cat not in legend_added
    legend_added.add(cat)
    hover = f"fatigue={seg['fatigue_factor']:.2f}"
    fig.add_trace(go.Scatter(
        x=dist_km[mask], y=ele[mask],
        mode='lines', line=dict(color=color, width=2.5),
        name=cat, legendgroup=cat, showlegend=show_leg,
        hovertemplate=f'<b>{cat}</b><br>%{{x:.1f}} km | %{{y:.0f}} m<br>'
                      f'{hover}<extra></extra>',
    ))

for nom, dist in RAVITOS:
    t_min = float(df_pred[df_pred['dist_end_km'] <= dist]['time_min'].sum())
    heure = (start_min + t_min) % (24 * 60)
    fig.add_vline(x=dist, line=dict(color='#f1c40f', width=1.5, dash='dot'))
    fig.add_annotation(
        x=dist, y=max(ele) * 1.02,
        text=f"<b>{nom}</b><br>{int(heure)//60:02d}h{int(heure)%60:02d}",
        showarrow=False, font=dict(size=10, color='#f1c40f'),
        bgcolor='rgba(0,0,0,0.5)', bordercolor='#f1c40f',
        borderwidth=1, xanchor='center',
    )

fig.update_layout(
    template='plotly_dark', title=f'Profil - {RACE_NAME}',
    xaxis_title='Distance (km)', yaxis_title='Altitude (m)',
    height=500, margin=dict(t=60, b=50, l=60, r=20),
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
)
fig.show()

## 7. Table des segments

In [32]:
cols = ['dist_start_km', 'dist_end_km', 'length_km',
        'dplus_m', 'dmoins_m', 'slope_mean_pct',
        'pace_min_per_km', 'speed_kmh', 'time_min']
df_d = df_pred[cols].copy()
df_d['pace_min_per_km'] = df_d['pace_min_per_km'].apply(format_pace)
df_d['time_min']        = df_d['time_min'].apply(format_time)
df_d = df_d.rename(columns={
    'dist_start_km': 'De (km)', 'dist_end_km': 'A (km)',
    'length_km': 'Dist (km)', 'dplus_m': 'D+ (m)',
    'dmoins_m': 'D- (m)', 'slope_mean_pct': 'Pente (%)',
    'pace_min_per_km': 'Allure (/km)',
    'speed_kmh': 'Vitesse (km/h)', 'time_min': 'Temps',
})
display(df_d)

,De (km),A (km),Dist (km),D+ (m),D- (m),Pente (%),Allure (/km),Vitesse (km/h),Temps
0,0.000000,1.271085,1.271085,0.293560,13.405361,-1.031544,"7'42""",7.78,9min
1,1.271085,1.736756,0.465671,10.769533,0.000000,2.312690,"9'18""",6.44,4min
2,1.736756,3.669185,1.932429,109.125531,2.093495,5.538731,"11'05""",5.40,21min
3,3.669185,6.701304,3.032119,407.312847,0.000000,13.433273,"16'15""",3.69,49min
4,6.701304,9.165880,2.464576,11.300098,178.285714,-6.775429,"6'40""",9.00,16min
5,9.165880,13.084866,3.918986,593.663289,0.516509,15.135210,"18'06""",3.31,1h 11min
6,13.084866,14.366802,1.281936,9.354037,78.373325,-5.383989,"6'40""",9.00,8min
7,14.366802,15.508542,1.141740,51.440667,0.207911,4.487252,"11'22""",5.28,13min
8,15.508542,16.363663,0.855120,197.311540,0.000000,23.074123,"25'08""",2.39,21min
9,16.363663,18.345949,1.982287,38.566525,172.124551,-6.737574,"6'40""",9.00,13min


## 8. Decomposition par troncons

In [33]:
builder   = SplitsBuilder(ravitos=RAVITOS, barrieres=BARRIERES,
                          start_min=start_min)
df_splits = builder.build(df_pred)
display(df_splits)

,Troncon,Dist (km),D+ (m),D- (m),Temps,Vitesse (km/h),Allure (/km),Heure arr.,Barriere
0,Depart -> Joux Plane,18.3,1428,441,3h 48min,4.8,"12'28""",07h48,8:00 (+11min)
1,Joux Plane -> Golèse,11.7,1073,985,2h 37min,4.5,"13'25""",10h25,
2,Golèse -> Refuge Bostan,8.1,651,841,1h 48min,4.5,"13'25""",12h14,
3,Refuge Bostan -> Vallon,8.5,128,927,1h 07min,7.5,"7'59""",13h22,16:00 (+157min)
4,Vallon -> Sixt,12.6,814,841,2h 50min,4.4,"13'33""",16h12,20:00 (+227min)
5,Sixt -> Arrivée,12.1,950,455,3h 04min,3.9,"15'17""",19h17,23:30 (+252min)


## 9. Heures de passage vs barrieres

In [34]:
labels    = [r.split('->')[-1].strip() for r in df_splits['Troncon']]
t_passage = [int(r['Heure arr.'][:2]) * 60 + int(r['Heure arr.'][3:])
             for _, r in df_splits.iterrows()]
labels_b  = [n for n, _, h in BARRIERES if h is not None]
t_barr    = [parse_hhmm(h) for _, _, h in BARRIERES if h is not None]

def fmt(m):
    return f'{int(m)//60:02d}h{int(m)%60:02d}'

tick_vals = list(range(int(min(t_passage + t_barr)) - 30,
                       int(max(t_passage + t_barr)) + 60, 60))
fig = go.Figure()
fig.add_trace(go.Bar(
    x=labels, y=t_passage, name='Passage predit',
    marker_color='#3498db',
    text=[fmt(t) for t in t_passage], textposition='outside',
))
fig.add_trace(go.Scatter(
    x=labels_b, y=t_barr, mode='markers+lines+text',
    name='Barriere', marker=dict(color='#e74c3c', size=10, symbol='diamond'),
    line=dict(color='#e74c3c', dash='dash'),
    text=[fmt(t) for t in t_barr], textposition='top center',
))
fig.update_layout(
    template='plotly_dark', title='Passage vs barrieres',
    yaxis=dict(tickvals=tick_vals, ticktext=[fmt(v) for v in tick_vals]),
    height=450, margin=dict(t=60, b=50),
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
)
fig.show()

## 10. Resume global

In [35]:
end_hhmm = f"{int(end_min) // 60 % 24:02d}h{int(end_min) % 60:02d}"
sep = '=' * 55
lines = [
    sep,
    f"  {summary['race_name']}",
    sep,
    f"  Distance    : {summary['distance_km']} km",
    f"  D+          : {summary['dplus_m']} m",
    f"  Depart      : {START_TIME}",
    f"  V_flat      : {V_FLAT_KMH} km/h (facteur {ATHLETE_FACTOR})",
    f"  Fatigue     : a={FATIGUE_A}  k={FATIGUE_K}  actif={USE_FATIGUE}",
    f"  Thermique   : seuil={T_THRESHOLD}C  actif={USE_THERMAL}",
    f"  Temps total : {summary['total_time_str']}",
    f"  Allure moy. : {summary['avg_pace_str']}/km",
    f"  Arrivee     : {end_hhmm}",
    sep,
]
print('\n'.join(lines))

  UTHG 2026
  Distance    : 75.85 km
  D+          : 5052 m
  Depart      : 04:00
  V_flat      : 9.0 km/h (facteur 0.85)
  Fatigue     : a=0.753  k=2.981  actif=True
  Thermique   : seuil=15.0C  actif=True
  Temps total : 15h 48min
  Allure moy. : 12'30"/km
  Arrivee     : 19h48


## 11. Export pacer Suunto (CSV)

In [36]:
exporter = PacerExporter(race_name=RACE_NAME)
csv_str  = exporter.to_csv(df_pred)
print(csv_str)

# UTHG 2026
distance_km;pace_str;time_cum_str;speed_kmh;dplus_m;slope_pct
1.27;"7'42""";9min;7.8;0;-1.0
1.74;"9'18""";14min;6.4;10;2.3
3.67;"11'05""";35min;5.4;109;5.5
6.7;"16'15""";1h 24min;3.7;407;13.4
9.17;"6'40""";1h 41min;9.0;11;-6.8
13.08;"18'06""";2h 52min;3.3;593;15.1
14.37;"6'40""";3h 00min;9.0;9;-5.4
15.51;"11'22""";3h 13min;5.3;51;4.5
16.36;"25'08""";3h 35min;2.4;197;23.1
18.35;"6'40""";3h 48min;9.0;38;-6.7
19.92;"22'56""";4h 24min;2.6;314;20.0
24.03;"6'40""";4h 52min;9.0;122;-15.5
26.62;"6'40""";5h 09min;9.0;30;-7.4
33.46;"22'29""";7h 43min;2.7;1223;17.9
40.71;"6'40""";8h 31min;9.0;49;-17.4
45.08;"6'40""";9h 00min;9.0;27;-9.5
45.87;"9'43""";9h 08min;6.2;1;-1.6
46.58;"18'33""";9h 21min;3.2;79;11.1
50.04;"24'25""";10h 45min;2.5;616;17.8
52.34;"7'03""";11h 02min;8.5;24;-7.6
52.91;"18'45""";11h 12min;3.2;67;11.4
56.28;"6'40""";11h 35min;9.0;0;-16.5
56.7;"27'49""";11h 47min;2.2;87;20.9
57.19;"6'40""";11h 50min;9.0;0;-15.9
59.53;"11'12""";12h 16min;5.3;24;0.7
63.38;"25'03""";13h 